# Correlation and Association

**Course:** Data Science  
**Lesson:** 05  
**Difficulty:** Beginner–Intermediate

## Learning Objectives

By the end of this notebook, students should be able to:

- explain covariance and correlation;
- calculate Pearson and Spearman correlation;
- interpret correlation strength and direction;
- create correlation matrices and heatmaps;
- examine relationships using scatter plots and regression lines;
- identify situations where correlation may be misleading;
- distinguish correlation from causation.

## Required Libraries

- Pandas
- NumPy
- Matplotlib
- Seaborn

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

## 1. Load the Dataset

This lesson reuses the synthetic employee dataset from the previous lesson.

In [ ]:
candidate_paths = [
    Path("../datasets/employee_statistics.csv"),
    Path("Data-Science-Course/datasets/employee_statistics.csv"),
    Path("datasets/employee_statistics.csv"),
]

data_path = next((path for path in candidate_paths if path.exists()), None)

if data_path is None:
    raise FileNotFoundError(
        "employee_statistics.csv was not found. "
        "Place it in Data-Science-Course/datasets/."
    )

df = pd.read_csv(data_path, parse_dates=["Hire_Date"])
df.head()

In [ ]:
df.shape

In [ ]:
df.info()

## 2. Select Numerical Variables

In [ ]:
numerical_columns = [
    "Age",
    "Years_Experience",
    "Annual_Salary_USD",
    "Weekly_Hours",
    "Performance_Score",
    "Job_Satisfaction",
    "Projects_Completed",
    "Training_Hours",
    "Absence_Days",
]

numeric_df = df[numerical_columns]
numeric_df.head()

## 3. Covariance

Covariance indicates whether two variables tend to move in the same or opposite directions.

- Positive covariance: both variables tend to increase together.
- Negative covariance: one tends to increase while the other decreases.
- Near-zero covariance: no clear linear co-movement.

Covariance depends on the units of the variables, so its magnitude is difficult to compare across variable pairs.

In [ ]:
covariance_value = df["Years_Experience"].cov(df["Annual_Salary_USD"])
covariance_value

In [ ]:
covariance_matrix = numeric_df.cov().round(2)
covariance_matrix

Covariance is useful for direction, but correlation is usually easier to interpret because it is standardized.

## 4. Pearson Correlation

Pearson correlation measures the strength and direction of a linear relationship between two numerical variables.

It ranges from -1 to +1.

In [ ]:
pearson_value = df["Years_Experience"].corr(
    df["Annual_Salary_USD"],
    method="pearson"
)
pearson_value

General interpretation:

- close to +1: strong positive linear relationship;
- close to -1: strong negative linear relationship;
- close to 0: weak or no linear relationship.

## 5. Correlation Strength Guide

The following guide is approximate and should not replace subject-matter judgment.

| Absolute correlation | Typical description |
|---:|---|
| 0.00–0.19 | Very weak |
| 0.20–0.39 | Weak |
| 0.40–0.59 | Moderate |
| 0.60–0.79 | Strong |
| 0.80–1.00 | Very strong |

In [ ]:
def describe_correlation(value):
    absolute_value = abs(value)

    if absolute_value < 0.20:
        strength = "very weak"
    elif absolute_value < 0.40:
        strength = "weak"
    elif absolute_value < 0.60:
        strength = "moderate"
    elif absolute_value < 0.80:
        strength = "strong"
    else:
        strength = "very strong"

    if value > 0:
        direction = "positive"
    elif value < 0:
        direction = "negative"
    else:
        direction = "no"

    return f"{strength} {direction} correlation"

describe_correlation(pearson_value)

## 6. Scatter Plot

A scatter plot should be examined before relying on a correlation coefficient.

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(
    df["Years_Experience"],
    df["Annual_Salary_USD"],
    alpha=0.65
)

plt.title("Years of Experience and Annual Salary")
plt.xlabel("Years of Experience")
plt.ylabel("Annual Salary (USD)")
plt.tight_layout()
plt.show()

The scatter plot shows the direction, shape, density, and potential outliers in the relationship.

## 7. Regression Line with Seaborn

In [ ]:
plt.figure(figsize=(8, 5))
sns.regplot(
    data=df,
    x="Years_Experience",
    y="Annual_Salary_USD",
    scatter_kws={"alpha": 0.55}
)

plt.title("Experience and Salary with Linear Trend")
plt.xlabel("Years of Experience")
plt.ylabel("Annual Salary (USD)")
plt.tight_layout()
plt.show()

The fitted line summarizes the average linear trend. Individual observations still vary around it.

## 8. Pearson Correlation Matrix

In [ ]:
pearson_matrix = numeric_df.corr(method="pearson").round(2)
pearson_matrix

A correlation matrix summarizes pairwise relationships among several numerical variables.

## 9. Correlation Heatmap

In [ ]:
plt.figure(figsize=(11, 8))
sns.heatmap(
    pearson_matrix,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0
)

plt.title("Pearson Correlation Matrix")
plt.tight_layout()
plt.show()

A heatmap makes strong positive and negative relationships easier to identify.

## 10. Strongest Correlations

In [ ]:
upper_triangle = pearson_matrix.where(
    np.triu(np.ones(pearson_matrix.shape), k=1).astype(bool)
)

strongest_pairs = (
    upper_triangle.stack()
    .rename("Correlation")
    .reset_index()
    .rename(columns={"level_0": "Variable_1", "level_1": "Variable_2"})
)

strongest_pairs["Absolute_Correlation"] = strongest_pairs["Correlation"].abs()

strongest_pairs.sort_values(
    "Absolute_Correlation",
    ascending=False
).head(10)

Ranking by absolute value identifies both strong positive and strong negative relationships.

## 11. Spearman Rank Correlation

Spearman correlation measures monotonic relationships using ranks.

It is useful when:

- the relationship is not linear;
- variables are ordinal;
- extreme values strongly affect Pearson correlation.

In [ ]:
spearman_value = df["Years_Experience"].corr(
    df["Annual_Salary_USD"],
    method="spearman"
)
spearman_value

In [ ]:
pd.DataFrame({
    "Method": ["Pearson", "Spearman"],
    "Correlation": [pearson_value, spearman_value]
}).round(3)

Similar Pearson and Spearman values suggest that the relationship is reasonably consistent and approximately linear. Large differences may indicate non-linearity or influential observations.

## 12. Spearman Correlation Matrix

In [ ]:
spearman_matrix = numeric_df.corr(method="spearman").round(2)
spearman_matrix

In [ ]:
plt.figure(figsize=(11, 8))
sns.heatmap(
    spearman_matrix,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0
)

plt.title("Spearman Correlation Matrix")
plt.tight_layout()
plt.show()

## 13. Pearson vs. Spearman Example

The following synthetic example shows a strong monotonic but non-linear relationship.

In [ ]:
x = np.arange(1, 21)
y = x ** 2

example_df = pd.DataFrame({"x": x, "y": y})

example_df.corr(method="pearson"), example_df.corr(method="spearman")

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(example_df["x"], example_df["y"])

plt.title("Monotonic but Non-Linear Relationship")
plt.xlabel("x")
plt.ylabel("y")
plt.tight_layout()
plt.show()

Spearman correlation can capture a consistently increasing relationship even when the relationship is not linear.

## 14. Correlation Within Groups

A relationship observed in the full dataset may differ across departments.

In [ ]:
department_correlations = (
    df.groupby("Department")
    .apply(
        lambda group: group["Years_Experience"].corr(
            group["Annual_Salary_USD"]
        ),
        include_groups=False
    )
    .rename("Experience_Salary_Correlation")
    .sort_values(ascending=False)
)

department_correlations

Group-level correlations can reveal differences that are hidden in the overall dataset.

## 15. Pair Plot

A pair plot displays pairwise relationships for a small selection of variables.

In [ ]:
selected_variables = [
    "Years_Experience",
    "Annual_Salary_USD",
    "Performance_Score",
    "Job_Satisfaction",
]

sns.pairplot(
    df[selected_variables],
    corner=True,
    diag_kind="hist"
)

plt.show()

Pair plots are useful for initial screening but become difficult to read when too many variables are included.

## 16. Correlation Does Not Imply Causation

A correlation between two variables does not prove that one causes the other.

Possible explanations include:

- direct causation;
- reverse causation;
- a third variable affecting both;
- coincidence;
- data collection bias.

For example, experience and salary may be positively correlated, but department, education level, job role, and seniority may also affect salary.

## 17. Outliers and Correlation

Extreme observations may change Pearson correlation substantially.

In [ ]:
full_correlation = df["Years_Experience"].corr(df["Annual_Salary_USD"])

q1 = df["Annual_Salary_USD"].quantile(0.25)
q3 = df["Annual_Salary_USD"].quantile(0.75)
iqr = q3 - q1
upper_bound = q3 + 1.5 * iqr

without_high_outliers = df[df["Annual_Salary_USD"] <= upper_bound]

filtered_correlation = without_high_outliers["Years_Experience"].corr(
    without_high_outliers["Annual_Salary_USD"]
)

pd.DataFrame({
    "Dataset": ["All observations", "High salary outliers excluded"],
    "Pearson_Correlation": [full_correlation, filtered_correlation]
}).round(3)

Removing outliers should never be automatic. Any exclusion must be justified and documented.

## 18. Categorical Association

Pearson and Spearman correlation are intended for numerical or ordinal variables.

Relationships between categorical variables require other methods, such as:

- contingency tables;
- chi-square tests;
- Cramér's V.

These methods are introduced in the hypothesis-testing lesson.

## 19. Practical Workflow

1. Define the analytical question.
2. Confirm that the variables are suitable.
3. Inspect missing values and data quality.
4. Create a scatter plot.
5. Calculate Pearson or Spearman correlation.
6. Check for outliers and group effects.
7. Interpret direction, strength, and context.
8. Avoid causal claims unless the research design supports them.

## 20. Common Mistakes

1. Interpreting correlation as causation.
2. Reporting only the coefficient without a plot.
3. Using Pearson correlation for strongly non-linear relationships.
4. Ignoring influential outliers.
5. Including identifiers in a correlation matrix.
6. Treating a weak correlation as proof of no relationship.
7. Ignoring subgroup differences.
8. Assuming high correlation means two variables are identical.

## 21. Exercises

1. Calculate Pearson correlation between `Age` and `Years_Experience`.
2. Calculate Spearman correlation between `Performance_Score` and `Projects_Completed`.
3. Create a scatter plot for `Weekly_Hours` and `Performance_Score`.
4. Interpret the direction and strength of that relationship.
5. Find the five strongest correlations in the numerical dataset.
6. Compare Pearson and Spearman correlation for `Training_Hours` and `Performance_Score`.
7. Calculate experience-salary correlation separately by gender.
8. Investigate whether high salary observations affect one correlation coefficient.
9. Create a heatmap using a subset of five numerical variables.
10. Explain why a strong correlation would not establish causation.

In [ ]:
# Write your exercise solutions here.

## 22. Summary

In this notebook, you learned how to:

- calculate covariance;
- calculate Pearson and Spearman correlation;
- interpret correlation strength and direction;
- create scatter plots, regression plots, matrices, and heatmaps;
- identify strong variable pairs;
- examine subgroup relationships;
- evaluate outlier influence;
- distinguish association from causation.

## Next Lesson

The next lesson is **Grouping, Aggregation, and Pivot Tables**.